<a href="https://colab.research.google.com/github/hellyaxs/MW-CL-pesquisa-tcc/blob/main/codigos/analise_estatistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise Estatística — Com ChatGPT vs. Sem ChatGPT

Este notebook executa os testes estatísticos principais do TCC:
- Estatísticas descritivas
- Teste de normalidade Shapiro–Wilk
- Teste de Mann–Whitney U (bicaudal)
- Cliff's Delta (tamanho de efeito)

## 1. Montar o Google Drive

Clique no link de autorização, cole o código e pressione Enter.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Caminho do arquivo

Edite a variável abaixo com o caminho completo do arquivo Excel no seu Drive.

**Exemplo:** `'/content/drive/MyDrive/TCC/Experimento - Elias.xlsx'`

In [ ]:
CAMINHO_ARQUIVO = "/content/drive/MyDrive/Colab Notebooks/codigo projeto tcc/Experimento - Elias.xlsx"  # <-- edite aqui

## 3. Instalar dependências

In [ ]:
!pip install -q openpyxl scipy

## 4. Código da análise

In [ ]:
from pathlib import Path
from statistics import mean, median, stdev

from openpyxl import load_workbook
from scipy.stats import mannwhitneyu, shapiro

ALPHA = 0.05
ABA_COM_GPT = "Usou ChatGPT"
ABA_SEM_GPT = "Não Usou"


def ler_pontuacoes(arquivo, aba):
    workbook = load_workbook(arquivo, data_only=True, read_only=True)
    if aba not in workbook.sheetnames:
        raise ValueError(
            f'A aba "{aba}" não foi encontrada. '
            f"Abas disponíveis: {', '.join(workbook.sheetnames)}"
        )
    pontuacoes = []
    worksheet = workbook[aba]
    for (valor,) in worksheet.iter_rows(
        min_row=2, min_col=2, max_col=2, values_only=True
    ):
        if valor is None:
            continue
        if isinstance(valor, str):
            valor = valor.split("/")[0].strip()
        try:
            pontuacoes.append(float(valor))
        except (TypeError, ValueError) as erro:
            raise ValueError(
                f'Pontuação inválida na aba "{aba}": {valor!r}'
            ) from erro
    if not pontuacoes:
        raise ValueError(f'Nenhuma pontuação foi encontrada na aba "{aba}".')
    return pontuacoes


def cliffs_delta(amostra_x, amostra_y):
    maiores = sum(x > y for x in amostra_x for y in amostra_y)
    menores = sum(x < y for x in amostra_x for y in amostra_y)
    return (maiores - menores) / (len(amostra_x) * len(amostra_y))


def magnitude_delta(delta):
    valor = abs(delta)
    if valor < 0.147:
        return "negligível"
    if valor < 0.330:
        return "pequeno"
    if valor < 0.474:
        return "médio"
    return "grande"


def formatar(numero, casas=3):
    return f"{numero:.{casas}f}".replace(".", ",")


def imprimir_descritivas(nome, valores):
    print(f"\n{nome}")
    print(f"  n: {len(valores)}")
    print(f"  Média: {formatar(mean(valores), 2)}")
    print(f"  Mediana: {formatar(median(valores), 2)}")
    print(f"  Desvio padrão amostral: {formatar(stdev(valores), 2)}")
    print(f"  Mínimo: {formatar(min(valores), 2)}")
    print(f"  Máximo: {formatar(max(valores), 2)}")


def executar(arquivo):
    com_gpt = ler_pontuacoes(arquivo, ABA_COM_GPT)
    sem_gpt = ler_pontuacoes(arquivo, ABA_SEM_GPT)

    print("=" * 68)
    print("ANÁLISE ESTATÍSTICA — COM CHATGPT vs. SEM CHATGPT")
    print(f"Arquivo: {arquivo}")
    print(f"Nível de significância: α = {formatar(ALPHA, 2)}")

    print("\n1. ESTATÍSTICAS DESCRITIVAS")
    imprimir_descritivas("Com ChatGPT", com_gpt)
    imprimir_descritivas("Sem ChatGPT", sem_gpt)

    print("\n2. SHAPIRO–WILK")
    for nome, valores in (
        ("Com ChatGPT", com_gpt),
        ("Sem ChatGPT", sem_gpt),
    ):
        resultado = shapiro(valores)
        decisao = (
            "rejeita normalidade"
            if resultado.pvalue < ALPHA
            else "não rejeita normalidade"
        )
        print(
            f"  {nome}: W = {formatar(resultado.statistic)}, "
            f"p = {formatar(resultado.pvalue)} → {decisao}"
        )

    print("\n3. MANN–WHITNEY U (bicaudal)")
    resultado_u = mannwhitneyu(
        com_gpt, sem_gpt,
        alternative="two-sided",
        method="asymptotic",
    )
    decisao_h0 = (
        "rejeita H₀" if resultado_u.pvalue < ALPHA else "não rejeita H₀"
    )
    print(f"  U = {formatar(resultado_u.statistic, 2)}")
    print(f"  p = {formatar(resultado_u.pvalue)}")
    print(f"  Decisão: {decisao_h0}")

    print("\n4. CLIFF'S DELTA")
    delta = cliffs_delta(com_gpt, sem_gpt)
    print(f"  δ = {formatar(delta)}")
    print(f"  |δ| = {formatar(abs(delta))}")
    print(f"  Magnitude: {magnitude_delta(delta)}")

    print("\n5. CONCLUSÃO")
    print(
        "  Não há evidência estatística de diferença mensurável "
        "entre as condições com e sem ChatGPT."
    )
    print("=" * 68)

## 5. Executar a análise

In [ ]:
executar(CAMINHO_ARQUIVO)

ANÁLISE ESTATÍSTICA — COM CHATGPT vs. SEM CHATGPT
Arquivo: /content/drive/MyDrive/Colab Notebooks/codigo projeto tcc/Experimento - Elias.xlsx
Nível de significância: α = 0,05

1. ESTATÍSTICAS DESCRITIVAS

Com ChatGPT
  n: 20
  Média: 15,05
  Mediana: 16,00
  Desvio padrão amostral: 3,43
  Mínimo: 6,00
  Máximo: 19,00

Sem ChatGPT
  n: 20
  Média: 14,90
  Mediana: 16,00
  Desvio padrão amostral: 3,97
  Mínimo: 7,00
  Máximo: 20,00

2. SHAPIRO–WILK
  Com ChatGPT: W = 0,875, p = 0,014 → rejeita normalidade
  Sem ChatGPT: W = 0,917, p = 0,088 → não rejeita normalidade

3. MANN–WHITNEY U (bicaudal)
  U = 199,50
  p = 1,000
  Decisão: não rejeita H₀

4. CLIFF'S DELTA
  δ = -0,003
  |δ| = 0,003
  Magnitude: negligível

5. CONCLUSÃO
  Não há evidência estatística de diferença mensurável entre as condições com e sem ChatGPT.
